# RVCBench — Protection + Qwen3-TTS Attack

End-to-end pipeline demonstrating:

1. **Protection** (`run_protect.py`): A protection algorithm perturbs the speaker's audio to make voice cloning harder, then evaluates *fidelity* (how audible the perturbation is).
2. **Attack** (`run_vc.py`): Qwen3-TTS zero-shot voice cloning is run against the dataset, producing *generation quality* metrics (MCD, WER, speaker similarity, MOS).

The default protection method here is **Gaussian Random Noise** (`grnoise_on_libritts`), which requires no external checkpoints and runs on any machine.  
To use **SafeSpeech** instead, swap `--config-name grnoise_on_libritts` → `safespeech_on_libritts` in Step 1 (requires BertVits2 checkpoints under `checkpoints/`).

**All outputs stay inside the repository directory.**

Requirements:
- GPU runtime (CUDA)
- `conda` environment named `qwen3` with `qwen-tts` installed, or run the pip install cell
- HF token with access to the Qwen3-TTS gated checkpoint

For exact gated download commands and SafeSpeech setup steps, see `docs/quickstart_model_setup.md`.

## 1. Locate the repository

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Nanboy-Ronan/RVCBench.git"

def _find_repo_root() -> Path:
    candidate = Path(os.getcwd()).resolve()
    for parent in [candidate] + list(candidate.parents):
        if (parent / "run_protect.py").exists():
            return parent
    return None

REPO_DIR = _find_repo_root()
if REPO_DIR is None:
    clone_target = Path(os.getcwd()) / "RVCBench"
    if not clone_target.exists():
        os.system(f"git clone {REPO_URL} {clone_target}")
    REPO_DIR = clone_target

%cd {REPO_DIR}
print("REPO_DIR:", REPO_DIR)
assert (REPO_DIR / "run_protect.py").exists(), "run_protect.py not found"
assert (REPO_DIR / "run_vc.py").exists(),      "run_vc.py not found"

## 2. Install dependencies

In [ ]:
%%bash
set -euxo pipefail
python -m pip install --upgrade pip
python -m pip install -q \
  hydra-core omegaconf pandas pyarrow \
  jiwer openai-whisper pymcd librosa soundfile \
  huggingface_hub
python -m pip install -q qwen-tts

## 3. Authenticate with Hugging Face

Qwen3-TTS weights are gated. Accept terms at `https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base` first.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token with access to Qwen3-TTS: ")

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF login configured.")

## 4. Download benchmark data into `{repo}/data/`

In [ ]:
from huggingface_hub import snapshot_download

HF_CONFIG  = "Libritts"
SPEAKER_ID = "1089"

DATA_DIR     = REPO_DIR / "data"
DATASET_ROOT = DATA_DIR / HF_CONFIG

snapshot_download(
    repo_id="Nanboy/RVCBench",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
    allow_patterns=[
        f"{HF_CONFIG}/metadata.parquet",
        f"{HF_CONFIG}/audios/{SPEAKER_ID}/*.wav",
    ],
)

wav_files = sorted((DATASET_ROOT / "audios" / SPEAKER_ID).glob("*.wav"))
print(f"Dataset root : {DATASET_ROOT}")
print(f"WAV files    : {len(wav_files)} for speaker {SPEAKER_ID}")

## 5. Configure both runs

In [ ]:
import torch

os.environ["DATA_LOADER_WORKERS"] = "0"

DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES = 5

# Protection config to use.
# - grnoise_on_libritts : Gaussian random noise, no checkpoints needed (default)
# - safespeech_on_libritts : SafeSpeech (requires BertVits2 checkpoints under checkpoints/)
PROTECT_CONFIG = "grnoise_on_libritts"
PROTECT_RUN_NAME = PROTECT_CONFIG  # used to locate results

print(f"REPO_DIR       = {REPO_DIR}")
print(f"DATASET_ROOT   = {DATASET_ROOT}")
print(f"DEVICE         = {DEVICE}")
print(f"PROTECT_CONFIG = {PROTECT_CONFIG}")
print(f"MAX_SAMPLES    = {MAX_SAMPLES}")

---
## Step 1 — Protection (`run_protect.py`)

Adds a perturbation to the speaker's audio, then measures fidelity.

Outputs written to `{REPO_DIR}/results/{PROTECT_CONFIG}/{timestamp}/`:
- `protected_audio/{speaker_id}/*.wav` — perturbed audio
- `fidelity_sample_metrics.csv` — per-file fidelity scores
- `metrics.json` — aggregated fidelity metrics

In [ ]:
%%bash -s "$PROTECT_CONFIG" "$DEVICE" "$DATASET_ROOT" "$SPEAKER_ID"
set -euxo pipefail

PROTECT_CONFIG=$1
DEVICE=$2
DATASET_ROOT=$3
SPEAKER_ID=$4

python run_protect.py \
  --config-name ${PROTECT_CONFIG} \
  device=${DEVICE} \
  dataset.root_path=${DATASET_ROOT} \
  dataset.use_hf_dataset=false \
  dataset.speaker_id=${SPEAKER_ID} \
  protection.batch_size=1

### Inspect protection results

In [ ]:
import json
from IPython.display import Audio, display

protect_results_dir = REPO_DIR / "results" / PROTECT_RUN_NAME
protect_run = max(protect_results_dir.glob("*"), key=lambda p: p.stat().st_mtime)
protect_metrics = json.loads((protect_run / "metrics.json").read_text())

print("Protection run dir:", protect_run)
print("\n=== Fidelity Metrics ===")
print(json.dumps(protect_metrics.get("fidelity_evaluation", {}), indent=2)[:2000])

PROTECTED_AUDIO_DIR = protect_run / "protected_audio"
protected_wavs = sorted(PROTECTED_AUDIO_DIR.rglob("*.wav"))
print(f"\nProtected audio files : {len(protected_wavs)}")
print(f"Protected audio dir   : {PROTECTED_AUDIO_DIR}")

In [ ]:
if protected_wavs:
    orig_path = DATASET_ROOT / "audios" / SPEAKER_ID / protected_wavs[0].name
    if orig_path.exists():
        print("Original audio:")
        display(Audio(str(orig_path)))
    print("Protected audio:")
    display(Audio(str(protected_wavs[0])))

---
## Step 2 — Voice Clone Attack (`run_vc.py` with Qwen3-TTS)

Qwen3-TTS performs zero-shot voice cloning using the dataset audio as the reference prompt.  
Comparing these generation metrics with the fidelity metrics above shows the protection's effectiveness.

Outputs written to `{REPO_DIR}/results/protect_qwen3tts_attack/{timestamp}/`.

In [ ]:
%%bash -s "$DEVICE" "$MAX_SAMPLES" "$DATASET_ROOT" "$SPEAKER_ID"
set -euxo pipefail

DEVICE=$1
MAX_SAMPLES=$2
DATASET_ROOT=$3
SPEAKER_ID=$4

python run_vc.py \
  --config-name ots_vc/clean/libritts/qwen3_tts_ots \
  run_name=protect_qwen3tts_attack \
  device=${DEVICE} \
  dataset.root_path=${DATASET_ROOT} \
  dataset.use_hf_dataset=false \
  dataset.speaker_id=${SPEAKER_ID} \
  adversary.max_samples=${MAX_SAMPLES} \
  batch_size=1

### Inspect voice cloning results

In [ ]:
vc_results_dir = REPO_DIR / "results" / "protect_qwen3tts_attack"
vc_run = max(vc_results_dir.glob("*"), key=lambda p: p.stat().st_mtime)
vc_metrics = json.loads((vc_run / "metrics.json").read_text())

print("VC run dir:", vc_run)
print("\n=== Generation Metrics ===")
print(json.dumps(vc_metrics.get("generation_evaluation", {}), indent=2)[:2000])

generated_wavs = sorted((vc_run / "generated_audio").rglob("*.wav"))
print(f"\nGenerated audio files: {len(generated_wavs)}")

In [ ]:
if generated_wavs:
    print("Cloned audio (Qwen3-TTS):")
    display(Audio(str(generated_wavs[0])))

---
## Summary: side-by-side metrics

In [ ]:
fid = protect_metrics.get("fidelity_evaluation", {})
gen = vc_metrics.get("generation_evaluation", {})

print("=" * 55)
print("STEP 1 — Fidelity (how perceptible is the protection?)")
print("=" * 55)
for key in ("avg_mcd", "avg_wer", "avg_speechmos_mos", "avg_dnsmos_ovrl", "avg_sim"):
    if key in fid:
        print(f"  {key:<28} {fid[key]:.4f}")

print()
print("=" * 55)
print("STEP 2 — Generation (how well does the attacker clone?)")
print("=" * 55)
for key in ("avg_mcd", "avg_wer", "avg_sim", "avg_speechmos_mos", "avg_dnsmos_ovrl", "emotion_match_rate"):
    if key in gen:
        print(f"  {key:<28} {gen[key]:.4f}")

print()
print("Lower MCD / WER = better quality. Higher SIM = more speaker-like.")
print("A good protection keeps step 1 MOS high while pushing step 2 MCD↑ and SIM↓.")

## Tips & common overrides

| Goal | How |
|---|---|
| Use SafeSpeech instead of grnoise | `PROTECT_CONFIG = "safespeech_on_libritts"` (needs BertVits2 checkpoints) |
| Stronger grnoise perturbation | `protection.epsilon=0.1` |
| More VC samples | `adversary.max_samples=50` |
| Different VC model | swap config to `f5_tts_ots`, `xtts_ots`, etc. |
| Different speaker | `dataset.speaker_id=1272` |
| Full dataset (auto HF) | remove `dataset.use_hf_dataset=false` and `dataset.root_path` overrides |
| Skip protection, re-evaluate | `protection.evaluate_only=true protection.evaluation.protected_audio_dir=<path>` |

See `configs/grnoise_on_libritts.yaml` and `configs/ots_vc/clean/libritts/qwen3_tts_ots.yaml` for all parameters.